# 第23章 数据合并与结构转换

通过merge、concat、melt、stack和unstack整合并重塑表格。


## 先解决一个小问题

拿一组小型业务数据练习“数据合并与结构转换”：先看数据结构，再完成一次明确的计算或转换。通过merge、concat、melt、stack和unstack整合并重塑表格。


## 这章为什么先学

这是“Pandas”路线中第 23 章的操作重点。本章只解决“数据合并与结构转换”，不重复前面章节已经完成的准备工作。


## 开始前确认

- 掌握 Python 基础语法、列表和字典
- 开始前先确认：按键连接表格


## 做完要留下什么

产出一个与“数据合并与结构转换”直接对应的结果，并记录输入形状、字段或筛选口径。


## 运行规则

代码单元格按依赖顺序执行；需要复现结果时从上到下运行，并保留输入、计算和输出。


## 本章要会

- 按键连接表格
- 纵向与横向拼接
- 宽表转长表
- 在索引层级间重塑


## 核心概念

- 连接前检查键唯一性和匹配率。
- concat负责轴向拼接，merge负责关系连接。
- 长表更适合分组统计和可视化。


## 示例 1：关系连接

validate参数可以验证预期的一对一或多对一关系。


In [ ]:
import pandas as pd


orders = pd.DataFrame({
    "order_id": ["A1", "A2", "A3"],
    "customer_id": ["U1", "U2", "U1"],
    "amount": [320, 880, 460],
})
customers = pd.DataFrame({
    "customer_id": ["U1", "U2"],
    "city": ["上海", "广州"],
})
merged = orders.merge(customers, on="customer_id", how="left", validate="many_to_one", indicator=True)
print(merged)


## 示例 2：数据拼接

拼接后通常需要重新建立连续索引。


In [ ]:
january = pd.DataFrame({"month": ["1月", "1月"], "sales": [120, 98]})
february = pd.DataFrame({"month": ["2月", "2月"], "sales": [150, 132]})
combined = pd.concat([january, february], ignore_index=True)
print(combined)


## 示例 3：宽表转长表

melt明确保留标识列，把多个指标列折叠为变量和值。


In [ ]:
wide = pd.DataFrame({
    "region": ["华东", "华南"],
    "一月": [120, 98],
    "二月": [150, 132],
    "三月": [180, 145],
})
long = wide.melt(id_vars="region", var_name="month", value_name="sales")
restored = long.pivot(index="region", columns="month", values="sales")
print(long)
print(restored)


## 公开大型数据实战

下面使用 UCI Machine Learning Repository 的 Online Retail 公开数据集。原始数据包含 541,909 条英国在线零售交易，本课程使用固定随机种子抽取的 200,000 行子集。分析时在完整子集上计算，只展示摘要或少量样本。


In [ ]:
import numpy as np
import pandas as pd


# UCI Machine Learning Repository: Online Retail
# 原始数据 541,909 行；课程使用固定随机种子抽取的 200,000 行子集。
data_url = "/datasets/uci_online_retail_200k.csv"
large_orders = pd.read_csv(
    data_url,
    parse_dates = ["InvoiceDate"],
    dtype = {"InvoiceNo": "string", "StockCode": "string", "Description": "string", "Country": "category"},
).rename(columns={
    "InvoiceNo": "order_id", "StockCode": "stock_code", "Description": "description",
    "Quantity": "quantity", "InvoiceDate": "order_time", "UnitPrice": "unit_price",
    "CustomerID": "customer_id", "Country": "country",
})
large_orders["sales"] = (large_orders["quantity"] * large_orders["unit_price"]).round(2)
large_orders["status"] = np.where(
    large_orders["order_id"].str.startswith("C") | (large_orders["quantity"] < 0),
    "取消/退货", "完成"
)
print(f"UCI Online Retail 公开数据：{len(large_orders):,} 行 × {large_orders.shape[1]} 列")
print("内存占用：", f"{large_orders.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
large_orders.head()


In [ ]:
customer_dimension = (
    large_orders.dropna(subset=["customer_id"])
    .groupby("customer_id", as_index=False)
    .agg(home_country=("country", "first"), first_order=("order_time", "min"))
)
enriched = large_orders.merge(customer_dimension, on="customer_id", how="left", validate="many_to_one", indicator=True)
country_sales = enriched.groupby("home_country", observed=True).agg(
    订单行数=("order_id", "size"), 销售额=("sales", "sum"), 客户数=("customer_id", "nunique")
).sort_values("销售额", ascending=False)
print("连接结果：", enriched.shape, "缺失客户维度：", (enriched["_merge"] != "both").sum())
display(country_sales.head(10).round(2))


## 常见误区

- 连接键不唯一导致行数意外膨胀
- 连接后不检查未匹配记录
- 重塑时遗漏标识列


## 综合练习

1. 连接订单表与商品表
2. 计算订单行金额
3. 将月度宽表转换为长表

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“连接订单表与商品表”。
2. **独立完成**：不复制示例代码，完成“计算订单行金额”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“将月度宽表转换为长表”，用一两句话说明你修改了什么。

### 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


In [ ]:
import pandas as pd


items = pd.DataFrame({"product_id": ["P1", "P2"], "price": [299, 129]})
order_lines = pd.DataFrame({
    "order_id": ["A1", "A1", "A2"],
    "product_id": ["P1", "P2", "P2"],
    "quantity": [1, 2, 3],
})

# TODO: 连接订单表与商品表
result = order_lines.merge(items, on="product_id", validate="many_to_one")

# TODO: 计算订单行金额
result["line_amount"] =

print(result)
print(result.groupby("order_id")["line_amount"].sum())


In [ ]:
import pandas as pd


items = pd.DataFrame({"product_id": ["P1", "P2"], "price": [299, 129]})
order_lines = pd.DataFrame({
    "order_id": ["A1", "A1", "A2"],
    "product_id": ["P1", "P2", "P2"],
    "quantity": [1, 2, 3],
})
result = order_lines.merge(items, on="product_id", validate="many_to_one")
result["line_amount"] = result["price"] * result["quantity"]
print(result)
print(result.groupby("order_id")["line_amount"].sum())

# 自检
assert len(result) == 3, "检查连接结果：应该有3行"
assert result["line_amount"].sum() == 686, "检查总金额：299 + 258 + 387"


## 本章小结

通过merge、concat、melt、stack和unstack整合并重塑表格。

**迁移思考**：

1. 如果订单表和商品表连接后行数突然增加了10倍，最可能的原因是什么？如何诊断？
2. 为什么长表更适合分组统计和可视化？宽表适合什么场景？


### 你已经掌握

- 按键连接表格
- 纵向与横向拼接
- 宽表转长表
- 在索引层级间重塑


### 验收标准

- 输入、计算和输出单元格完整。
- 关键变量类型、形状或数值可核对。
- 结论引用输出证据，并注明适用范围。


### 关键知识速查

| 知识点 | 作用与提醒 | 关键写法 |
| --- | --- | --- |
| 关系连接 | validate参数可以验证预期的一对一或多对一关系。 | `pd.DataFrame()`、`orders.merge()` |
| 数据拼接 | 拼接后通常需要重新建立连续索引。 | `pd.DataFrame()`、`pd.concat()` |
| 宽表转长表 | melt明确保留标识列，把多个指标列折叠为变量和值。 | `pd.DataFrame()`、`wide.melt()`、`long.pivot()` |


### 需要注意

- 连接键不唯一导致行数意外膨胀
- 连接后不检查未匹配记录
- 重塑时遗漏标识列


### 完成检查

- [ ] 能够按键连接表格
- [ ] 能够纵向与横向拼接
- [ ] 能够宽表转长表
- [ ] 能够在索引层级间重塑


### 排错顺序

1. 从上到下重新运行依赖单元格。
2. 检查变量类型、列名、形状和缺失值。
3. 缩小输入范围，定位产生错误的最小步骤。
4. 修复后重新运行完整流程。
